In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.decomposition import PCA
import json
import warnings
warnings.filterwarnings('ignore')

# Фиксируем random state для воспроизводимости
RANDOM_STATE = 42

# Создаем структуру папок согласно требованиям задания
DATA_DIR = Path('data')
ARTIFACTS_DIR = Path('artifacts')
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
LABELS_DIR = ARTIFACTS_DIR / 'labels'

ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
LABELS_DIR.mkdir(exist_ok=True)

# Словари для хранения результатов
all_results = {}
best_configs = {}

# === ЗАГРУЗКА И ОБРАБОТКА ДАННЫХ ===
# Используем 3 датасета из 4 возможных согласно заданию S07-homework:
# S07-hw-dataset-01.csv - данные с признаками в разных шкалах
# S07-hw-dataset-02.csv - данные с нелинейной структурой и выбросами  
# S07-hw-dataset-03.csv - данные с кластерами разной плотности
# S07-hw-dataset-04.csv - НЕ ИСПОЛЬЗУЕТСЯ, так как содержит категориальные признаки,
# а в данном задании мы фокусируемся на числовых данных для демонстрации базовых алгоритмов

datasets = [
    'S07-hw-dataset-01',  # Файл: S07-hw-dataset-01.csv
    'S07-hw-dataset-02',  # Файл: S07-hw-dataset-02.csv  
    'S07-hw-dataset-03'   # Файл: S07-hw-dataset-03.csv
]

for ds_name in datasets:
    print(f"=== Обработка датасета: {ds_name}.csv ===")
    
    # === 2.3.1 Загрузка данных и первичный анализ ===
    # Загружаем CSV файл: S07-hw-dataset-0X.csv
    df = pd.read_csv(DATA_DIR / f'{ds_name}.csv')
    
    # Выводим информацию о данных для отчета
    print(f"Размер данных: {df.shape}")
    print("\nПервые 5 строк данных:")
    print(df.head())
    print("\nИнформация о типах данных:")
    print(df.info())
    print("\nБазовые статистики:")
    print(df.describe())
    
    # Извлекаем sample_id и признаки
    sample_ids = df['sample_id']
    X_raw = df.drop(columns=['sample_id'])
    
    # === 2.3.2 Препроцессинг ===
    # Проверяем наличие нечисловых признаков
    numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric_cols = [col for col in X_raw.columns if col not in numeric_cols]
    
    if len(non_numeric_cols) > 0:
        print(f"  В датасете {ds_name}.csv обнаружены нечисловые признаки: {non_numeric_cols}")
        print("   Используем только числовые признаки для данного задания.")
        print("   ПРИМЕЧАНИЕ: Для обработки категориальных признаков (как в S07-hw-dataset-04.csv) ")
        print("   необходимо использовать OneHotEncoder, но в выбранных датасетах его нет.")
        X_raw = X_raw[numeric_cols]
    else:
        print(f" Все признаки в {ds_name}.csv являются числовыми.")
    
    # Проверка и обработка пропусков
    if X_raw.isnull().sum().sum() > 0:
        print(f"  Обнаружены пропуски в {ds_name}.csv. Применяем SimpleImputer.")
        imputer = SimpleImputer(strategy='median')
        X_imputed = imputer.fit_transform(X_raw)
    else:
        print(f" Пропусков в {ds_name}.csv не обнаружено.")
        X_imputed = X_raw.values
    
    # Масштабирование - ОБЯЗАТЕЛЬНЫЙ шаг для distance-based методов
    print("Применяем StandardScaler для масштабирования признаков...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)
    
    # === 2.3.3 Модели кластеризации ===
    print("\n=== Обучение KMeans ===")
    # KMeans (обязательный алгоритм)
    k_range = range(2, 11)
    sil_scores = []
    
    print(f"Подбираем оптимальное k для KMeans на {ds_name}.csv...")
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        sil_score = silhouette_score(X_scaled, labels)
        sil_scores.append(sil_score)
        print(f"  k={k}: silhouette_score = {sil_score:.4f}")
    
    best_k = k_range[np.argmax(sil_scores)]
    print(f" Лучшее k для KMeans на {ds_name}.csv: {best_k} (silhouette={max(sil_scores):.4f})")
    
    # Обучаем лучшую модель KMeans
    kmeans_best = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
    labels_kmeans = kmeans_best.fit_predict(X_scaled)
    
    print("\n=== Обучение DBSCAN ===")
    # DBSCAN (второй обязательный алгоритм)
    min_samples = max(5, int(0.01 * len(X_scaled)))
    eps_candidates = [0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
    best_sil_db = -1
    best_eps = None
    best_labels_db = None
    
    print(f"Подбираем параметры DBSCAN для {ds_name}.csv (min_samples={min_samples})...")
    for eps in eps_candidates:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels_db = dbscan.fit_predict(X_scaled)
        n_noise = np.sum(labels_db == -1)
        noise_ratio = n_noise / len(labels_db)
        unique_labels = set(labels_db)
        
        print(f"  eps={eps:.1f}: кластеров={len(unique_labels)-1 if -1 in unique_labels else len(unique_labels)}, "
              f"шум={noise_ratio:.2%}")
        
        # Пропускаем плохие конфигурации
        if noise_ratio > 0.6 or len(unique_labels) < 2 or (-1 in unique_labels and len(unique_labels) == 2):
            continue
        
        try:
            # Считаем метрики только на non-noise точках
            non_noise_mask = labels_db != -1
            if np.sum(non_noise_mask) > 1:  # Нужно минимум 2 точки для silhouette
                sil = silhouette_score(X_scaled[non_noise_mask], labels_db[non_noise_mask])
                print(f"    silhouette_score (без шума) = {sil:.4f}")
                
                if sil > best_sil_db:
                    best_sil_db = sil
                    best_eps = eps
                    best_labels_db = labels_db.copy()
        except Exception as e:
            print(f"    Ошибка при расчете метрик для eps={eps}: {e}")
            continue
    
    if best_labels_db is not None:
        print(f" Лучшие параметры DBSCAN для {ds_name}.csv: eps={best_eps}, silhouette={best_sil_db:.4f}")
    else:
        print(f" Не удалось найти подходящую конфигурацию DBSCAN для {ds_name}.csv")
    
    # === 2.3.4 Метрики качества ===
    res = {}
    
    # Метрики для KMeans
    res['KMeans'] = {
        'k': int(best_k),
        'silhouette': float(silhouette_score(X_scaled, labels_kmeans)),
        'davies_bouldin': float(davies_bouldin_score(X_scaled, labels_kmeans)),
        'calinski_harabasz': float(calinski_harabasz_score(X_scaled, labels_kmeans)),
        'noise_ratio': 0.0
    }
    
    # Метрики для DBSCAN (если найдена хорошая конфигурация)
    if best_labels_db is not None:
        non_noise = best_labels_db != -1
        if np.sum(non_noise) > 1:  # Проверка, что есть non-noise точки
            res['DBSCAN'] = {
                'eps': float(best_eps),
                'min_samples': int(min_samples),
                'silhouette': float(silhouette_score(X_scaled[non_noise], best_labels_db[non_noise])),
                'davies_bouldin': float(davies_bouldin_score(X_scaled[non_noise], best_labels_db[non_noise])),
                'calinski_harabasz': float(calinski_harabasz_score(X_scaled[non_noise], best_labels_db[non_noise])),
                'noise_ratio': float(np.mean(best_labels_db == -1))
            }
        else:
            res['DBSCAN'] = None
    else:
        res['DBSCAN'] = None
    
    # === 2.3.7 Выбор лучшего метода ===
    best_method = 'KMeans'
    best_labels = labels_kmeans
    
    if res['DBSCAN'] and res['DBSCAN']['silhouette'] > res['KMeans']['silhouette']:
        best_method = 'DBSCAN'
        best_labels = best_labels_db
        print(f" Выбран DBSCAN как лучший метод для {ds_name}.csv")
    else:
        print(f" Выбран KMeans как лучший метод для {ds_name}.csv")
    
    # === 2.3.5 Визуализация ===
    print("\n=== Создание визуализаций ===")
    # PCA для визуализации
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)
    
    # PCA scatter plot
    plt.figure(figsize=(8, 6))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, cmap='tab10', alpha=0.7, s=30)
    plt.title(f'{ds_name}.csv: PCA + {best_method}', fontsize=14, fontweight='bold')
    plt.xlabel('PCA Component 1', fontsize=12)
    plt.ylabel('PCA Component 2', fontsize=12)
    plt.colorbar(label='Cluster Label')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'pca_{ds_name}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Сохранен график PCA: artifacts/figures/pca_{ds_name}.png")
    
    # Silhouette vs k plot
    plt.figure(figsize=(10, 6))
    plt.plot(k_range, sil_scores, 'bo-', linewidth=2, markersize=8)
    plt.axvline(best_k, color='r', linestyle='--', label=f'Best k={best_k}', linewidth=2)
    plt.title(f'{ds_name}.csv: Silhouette Score vs Number of Clusters (k)', fontsize=14, fontweight='bold')
    plt.xlabel('Number of clusters (k)', fontsize=12)
    plt.ylabel('Silhouette Score', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'silhouette_vs_k_{ds_name}.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Сохранен график Silhouette vs k: artifacts/figures/silhouette_vs_k_{ds_name}.png")
    
    # === 2.3.6 и 2.4 Сохранение результатов ===
    print("\n=== Сохранение результатов ===")
    # Сохранение меток кластеров
    output_df = pd.DataFrame({
        'sample_id': sample_ids,
        'cluster_label': best_labels
    })
    
    labels_filename = f'labels_hw07_{ds_name.split("-")[-1]}.csv'
    output_df.to_csv(LABELS_DIR / labels_filename, index=False)
    print(f"Сохранены метки кластеров: artifacts/labels/{labels_filename}")
    
    # Сохранение в словари для итоговой сводки
    all_results[ds_name] = res
    best_configs[ds_name] = {
        'best_method': best_method,
        'params': res[best_method],
        'filename': f'{ds_name}.csv'
    }
    
    print(f"=== Обработка {ds_name}.csv завершена ===\n")
    print("=" * 60 + "\n")

# === 2.3.6 Проверка устойчивости (только для одного датасета) ===
print("\n" + "="*70)
print("=== ПРОВЕРКА УСТОЙЧИВОСТИ KMEANS НА DATASET-01 ===")
print("="*70)

# Загружаем Dataset-01 снова для проверки устойчивости
ds_name_stability = 'S07-hw-dataset-01'
print(f"Загружаем {ds_name_stability}.csv для проверки устойчивости...")
df_stability = pd.read_csv(DATA_DIR / f'{ds_name_stability}.csv')
X_stability = df_stability.drop(columns=['sample_id'])

# Препроцессинг (тот же самый)
X_stability_scaled = StandardScaler().fit_transform(X_stability)

# Используем k из результатов основного эксперимента
k_stability = best_configs['S07-hw-dataset-01']['params']['k']
print(f"Проверяем устойчивость KMeans с k={k_stability} на {ds_name_stability}.csv")

ari_list = []
ref_labels = KMeans(n_clusters=k_stability, random_state=42, n_init=10).fit_predict(X_stability_scaled)

print("Запуск 5 итераций с разными random_state...")
for seed in range(5):
    new_labels = KMeans(n_clusters=k_stability, random_state=seed, n_init=10).fit_predict(X_stability_scaled)
    ari = adjusted_rand_score(ref_labels, new_labels)
    ari_list.append(ari)
    print(f"  Iteration {seed}: random_state={seed}, ARI={ari:.4f}")

mean_ari = np.mean(ari_list)
std_ari = np.std(ari_list)

print(f"\nРезультаты устойчивости для {ds_name_stability}.csv:")
print(f"   ARI между запусками: {[round(x, 4) for x in ari_list]}")
print(f"   Среднее ARI: {mean_ari:.4f} ± {std_ari:.4f}")

# Визуализация устойчивости
plt.figure(figsize=(10, 6))
plt.bar(range(5), ari_list, color='skyblue', alpha=0.8)
plt.axhline(y=mean_ari, color='r', linestyle='--', label=f'Среднее ARI = {mean_ari:.4f}')
plt.title(f'Устойчивость KMeans на {ds_name_stability}.csv (k={k_stability})', fontsize=14, fontweight='bold')
plt.xlabel('Номер итерации', fontsize=12)
plt.ylabel('Adjusted Rand Index (ARI)', fontsize=12)
plt.ylim(0.9, 1.01)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'stability_{ds_name_stability}.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Сохранен график устойчивости: artifacts/figures/stability_{ds_name_stability}.png")

# === 2.4 Сохранение артефактов ===
print("\n" + "="*70)
print("=== СОХРАНЕНИЕ ФИНАЛЬНЫХ АРТЕФАКТОВ ===")
print("="*70)

# metrics_summary.json
metrics_path = ARTIFACTS_DIR / 'metrics_summary.json'
with open(metrics_path, 'w') as f:
    json.dump(all_results, f, indent=4)
print(f"Сохранена сводка метрик: {metrics_path}")

# best_configs.json
configs_path = ARTIFACTS_DIR / 'best_configs.json'
with open(configs_path, 'w') as f:
    json.dump(best_configs, f, indent=4)
print(f"Сохранены лучшие конфигурации: {configs_path}")



=== Обработка датасета: S07-hw-dataset-01.csv ===
Размер данных: (12000, 9)

Первые 5 строк данных:
   sample_id        f01        f02       f03         f04        f05  \
0          0  -0.536647 -69.812900 -0.002657   71.743147 -11.396498   
1          1  15.230731  52.727216 -1.273634 -104.123302  11.589643   
2          2  18.542693  77.317150 -1.321686 -111.946636  10.254346   
3          3 -12.538905 -41.709458  0.146474   16.322124   1.391137   
4          4  -6.903056  61.833444 -0.022466  -42.631335   3.107154   

         f06        f07       f08  
0 -12.291287  -6.836847 -0.504094  
1  34.316967 -49.468873  0.390356  
2  25.892951  44.595250  0.325893  
3   2.014316 -39.930582  0.139297  
4  -5.471054   7.001149  0.131213  

Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 